# 데이터 전처리 Preprocessing
- 데이터 클렌징: 결측치/중복값/이상치 처리
- 데이터 인코딩: 문자값 수치로 변환 (LabelEncoding, OneHotEncoding)
- 데이터 스케일링: 표준정규화, 최대/최소정규화, RobustScaler. KNN/SVM/KMeans처럼 거리나 크기를 보는 모델에서 큰 숫자 feature가 판단을 지배하지 않게 하기 위함.
- 특성공학:
  - 특성선택/추출
  - 가공
  - 다항처리


In [ ]:
# 초기 세팅용 import 구문입니다. 먼저 실행한 뒤 실습 코드를 작성합니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.datasets import load_iris


## 데이터 클렌징
- 결측치/중복값/이상치 처리



#### `SimpleImputer`: 결측치를 평균, 중앙값, 최빈값 등 지정한 전략으로 대체함.


In [ ]:
data = pd.DataFrame({
    "age": [25, None, 27, None, 30],
    "income": [50000, 60000, None, 52000, 58000]
})

#SimplrImputer() : 결측치 전저리

imputer = SimpleImputer(strategy='median', copy=False) # 결측치 mean으로 변환
data_cleaned = pd.DataFrame(
    imputer.fit_transform(data),
    columns=data.columns
)

data_cleaned

'IsolationForest()' : 트리구조 이상치 처리

In [ ]:
# 이상치 처리

from sklearn.ensemble import IsolationForest

X = np.array([[1], [2], [2.5], [3], [100]])

iso = IsolationForest(random_state=2)

pred = iso.fit_predict(X) # 1또는 -1

print(pred)

X_cleaned = X[pred == 1]

pred(X_cleaned)

## 데이터 인코딩 - 범주형 인코딩
- 문자 범주를 모델이 계산할 수 있는 숫자 표현으로 바꾸는 과정.


#### `LabelEncoder`: 문자 범주를 0, 1, 2 같은 정수 라벨로 변환함.


In [18]:
from sklearn.preprocessing import LabelEncoder

items = ['TV', '냉장고', '전자렌지', '컴퓨터', '선풍기', '선풍기', '믹서', '믹서']

encoder = LabelEncoder()

encoder.fit(items) # 데이터 학습

labels = encoder.transform(items)

print(labels)
print(encoder.classes_)

[0 1 4 5 3 3 2 2]
['TV' '냉장고' '믹서' '선풍기' '전자렌지' '컴퓨터']


In [19]:
# 복원

encoder.inverse_transform([2,1,0])


array(['믹서', '냉장고', 'TV'], dtype='<U4')

#### One-hot Encoding : 범주형 변수마다 하나의 컬럼을 생성하고, 해당 범주에만 1을 표시하는 인코딩
#### (one vs. rest 또는 one vs. all)


In [23]:
from sklearn.preprocessing import OneHotEncoder

items = ['TV', '냉장고', '전자렌지', '컴퓨터', '선풍기', '선풍기', '믹서', '믹서']

items = np.array(items).reshape(-1,1) # reshape(-1,1) -> -1 알아서,
# print(items)

encoder = OneHotEncoder()
encoder.fit(items)

print(encoder.categories_)

[array(['TV', '냉장고', '믹서', '선풍기', '전자렌지', '컴퓨터'], dtype='<U4')]


In [27]:
one_hot_items = encoder.transform(items)
print(one_hot_items.toarray())



(8, 6)


## 피처 스케일링과 정규화
머신러닝에서 **피처 스케일링**은 서로 다른 범위를 가지는 데이터를 공통된 기준으로 맞추기 위한 과정이다.
### 표준화 (Standardization)
- **정의**: 평균을 0, 표준편차를 1로 변환하는 방식
- **공식**:
  $$
  z = \frac{x - \mu}{\sigma}
  $$
  여기서 $ \mu $는 평균, $ \sigma $는 표준편차
- **특징**:
  - 데이터가 **정규분포를 따를 때** 효과적
  - **음수 값도 허용**
  - 이상치(outlier)의 영향을 **덜 받음**

---

### 정규화 (Normalization, Min-Max Scaling)
- **정의**: 데이터를 0과 1 사이로 스케일링
- **공식**:
  $$
  x_{norm} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}
  $$
- **특징**:
  - **특정 범위(보통 0~1)**로 제한
  - **이상치에 민감**
  - 거리 기반 알고리즘(k-NN, SVM 등)에 자주 사용

---

### 로버스트 스케일링 (Robust Scaling)
- **정의**: 중앙값(Median)과 사분위범위(IQR)를 이용한 스케일링
- **공식**:
  $$
  x_{scaled} = \frac{x - \text{Median}}{\text{IQR}}
  $$
  여기서 IQR = Q3 - Q1 (제3사분위수 - 제1사분위수)
- **특징**:
  - **이상치에 강건(Robust)**
  - 중앙값과 IQR 기준 → **데이터 분포 왜곡이 적음**
  - **StandardScaler보다 이상치 영향이 적은 대안**
  - 다양한 분포의 데이터에 적용 가능


## 데이터 스케일링

#### StandardScaler: 평균 0, 표준편차 1 기준으로 수치 feature의 단위를 맞춤.


#### MinMaxScaler: 최소값 0, 최대값 1 범위로 수치 feature의 범위를 맞춤.


#### RobustScaler: 중앙값과 IQR을 기준으로 스케일링해 이상치 영향을 줄임.


## 데이터 스케일링

- StandardScaler: 평균 0, 표준편차 1 기준으로 수치 feature의 단위를 맞춤.
- MinMaxScaler: 최소값 0, 최대값 1 범위로 수치 feature의 범위를 맞춤.
- RobustScaler: 중앙값과 IQR을 기준으로 스케일링해 이상치 영향을 줄임.
